In [1]:
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

StatementMeta(, c427f369-47bc-4533-aa4d-0d042d3457bf, 3, Finished, Available, Finished, False)

DataFrame[]

In [2]:
from delta.tables import DeltaTable

StatementMeta(, c427f369-47bc-4533-aa4d-0d042d3457bf, 4, Finished, Available, Finished, False)

In [3]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window



StatementMeta(, c427f369-47bc-4533-aa4d-0d042d3457bf, 5, Finished, Available, Finished, False)

In [4]:
# --- crm_cust_info (MERGE) ---
df = spark.read.table("bronze.crm_cust_info")
w = Window.partitionBy("cst_id").orderBy(F.col("cst_create_date").desc())

silver_cust_info = (
    df.withColumn("rn", F.row_number().over(w))
    .filter((F.col("rn") == 1) & F.col("cst_id").isNotNull())
    .withColumn("cst_firstname", F.trim("cst_firstname"))
    .withColumn("cst_lastname", F.trim("cst_lastname"))
    .withColumn("cst_marital_status",
        F.when(F.upper(F.trim("cst_marital_status")) == "S", "Single")
         .when(F.upper(F.trim("cst_marital_status")) == "M", "Married")
         .otherwise("n/a"))
    .withColumn("cst_gndr",
        F.when(F.upper(F.trim("cst_gndr")) == "F", "Female")
         .when(F.upper(F.trim("cst_gndr")) == "M", "Male")
         .otherwise("n/a"))
    .drop("rn")
)

if spark.catalog.tableExists("silver.crm_cust_info"):
    target = DeltaTable.forName(spark, "silver.crm_cust_info")
    (target.alias("t")
        .merge(silver_cust_info.alias("s"), "t.cst_id = s.cst_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
else:
    silver_cust_info.write.format("delta").saveAsTable("silver.crm_cust_info")

StatementMeta(, c427f369-47bc-4533-aa4d-0d042d3457bf, 6, Finished, Available, Finished, False)

In [5]:
# --- crm_prd_info (MERGE) ---

w2 = Window.partitionBy("prd_key").orderBy("prd_start_dt")
df = spark.read.table("bronze.crm_prd_info")

silver_prd_info = (
    df.withColumn("cat_id", F.regexp_replace(F.substring("prd_key", 1, 5), "-", "_"))
    .withColumn("prd_key", F.expr("substring(prd_key, 7, length(prd_key))"))
    .withColumn("prd_cost", F.coalesce(F.col("prd_cost"), F.lit(0)))
    .withColumn("prd_line",
        F.when(F.upper(F.trim("prd_line")) == "M", "Mountain")
         .when(F.upper(F.trim("prd_line")) == "R", "Road")
         .when(F.upper(F.trim("prd_line")) == "S", "Other Sales")
         .when(F.upper(F.trim("prd_line")) == "T", "Touring")
         .otherwise("n/a"))
    .withColumn("prd_start_dt", F.to_date("prd_start_dt"))
    .withColumn("prd_end_dt", F.date_sub(F.lead("prd_start_dt").over(w2), 1))
    .select("prd_id", "cat_id", "prd_key", "prd_nm", "prd_cost", "prd_line", "prd_start_dt", "prd_end_dt")
)

# Dedupe on merge key before merging
w_dedupe_prd = Window.partitionBy("prd_id").orderBy(F.col("prd_start_dt").desc())
silver_prd_info = (
    silver_prd_info
    .withColumn("rn", F.row_number().over(w_dedupe_prd))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

if spark.catalog.tableExists("silver.crm_prd_info"):
    target = DeltaTable.forName(spark, "silver.crm_prd_info")
    (target.alias("t")
        .merge(silver_prd_info.alias("s"), "t.prd_id = s.prd_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
else:
    silver_prd_info.write.format("delta").saveAsTable("silver.crm_prd_info")

spark.read.table("silver.crm_prd_info").show(5)

StatementMeta(, c427f369-47bc-4533-aa4d-0d042d3457bf, 7, Finished, Available, Finished, False)

+------+------+----------+--------------------+--------+-----------+------------+----------+
|prd_id|cat_id|   prd_key|              prd_nm|prd_cost|   prd_line|prd_start_dt|prd_end_dt|
+------+------+----------+--------------------+--------+-----------+------------+----------+
|   210| CO_RF|FR-R92B-58|HL Road Frame - B...|       0|       Road|  2003-07-01|2003-06-30|
|   211| CO_RF|FR-R92R-58|HL Road Frame - R...|       0|       Road|  2003-07-01|2003-06-30|
|   212| AC_HE| HL-U509-R|Sport-100 Helmet-...|      12|Other Sales|  2011-07-01|2011-06-30|
|   213| AC_HE| HL-U509-R|Sport-100 Helmet-...|      14|Other Sales|  2012-07-01|2012-06-30|
|   214| AC_HE| HL-U509-R|Sport-100 Helmet-...|      13|Other Sales|  2013-07-01|2013-06-30|
+------+------+----------+--------------------+--------+-----------+------------+----------+
only showing top 5 rows



In [6]:
# --- crm_sales_details (MERGE) ---
def parse_yyyymmdd(col):
    return F.when(
        (F.col(col) == 0) | (F.length(F.col(col).cast("string")) != 8),
        None
    ).otherwise(F.to_date(F.col(col).cast("string"), "yyyyMMdd"))

df = spark.read.table("bronze.crm_sales_details")

silver_sales_details = (
    df.withColumn("sls_order_dt", parse_yyyymmdd("sls_order_dt"))
    .withColumn("sls_ship_dt", parse_yyyymmdd("sls_ship_dt"))
    .withColumn("sls_due_dt", parse_yyyymmdd("sls_due_dt"))
    .withColumn("sls_sales",
        F.when(
            F.col("sls_sales").isNull() | (F.col("sls_sales") <= 0) |
            (F.col("sls_sales") != F.col("sls_quantity") * F.abs(F.col("sls_price"))),
            F.col("sls_quantity") * F.abs(F.col("sls_price"))
        ).otherwise(F.col("sls_sales")))
    .withColumn("sls_price",
        F.when(F.col("sls_price").isNull() | (F.col("sls_price") <= 0),
            F.when(F.col("sls_quantity") != 0, F.col("sls_sales") / F.col("sls_quantity")).otherwise(None)
        ).otherwise(F.col("sls_price")))
)

# Dedupe on merge key before merging
w_dedupe_sales = Window.partitionBy("sls_ord_num", "sls_prd_key").orderBy(F.col("sls_order_dt").desc())
silver_sales_details = (
    silver_sales_details
    .withColumn("rn", F.row_number().over(w_dedupe_sales))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

if spark.catalog.tableExists("silver.crm_sales_details"):
    target = DeltaTable.forName(spark, "silver.crm_sales_details")
    (target.alias("t")
        .merge(silver_sales_details.alias("s"), "t.sls_ord_num = s.sls_ord_num AND t.sls_prd_key = s.sls_prd_key")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
else:
    silver_sales_details.write.format("delta").saveAsTable("silver.crm_sales_details")

spark.read.table("silver.crm_sales_details").show(5)

StatementMeta(, c427f369-47bc-4533-aa4d-0d042d3457bf, 8, Finished, Available, Finished, False)

+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+
|sls_ord_num|sls_prd_key|sls_cust_id|sls_order_dt|sls_ship_dt|sls_due_dt|sls_sales|sls_quantity|sls_price|
+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+
|    SO43697| BK-R93R-62|      21768|  2010-12-29| 2011-01-05|2011-01-10|     3578|           1|   3578.0|
|    SO43705| BK-M82S-38|      11011|  2010-12-30| 2011-01-06|2011-01-11|     3400|           1|   3400.0|
|    SO43715| BK-R93R-56|      16483|  2011-01-02| 2011-01-09|2011-01-14|     3578|           1|   3578.0|
|    SO43717| BK-R50B-62|      25249|  2011-01-02| 2011-01-09|2011-01-14|      699|           1|    699.0|
|    SO43722| BK-R93R-48|      13581|  2011-01-04| 2011-01-11|2011-01-16|     3578|           1|   3578.0|
+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+
only showing top 5 rows



In [7]:
df = spark.read.table("bronze.erp_cust_az12")

silver_erp_cust = (
    df.withColumn("cid",
        F.when(F.col("cid").startswith("NAS"), F.expr("substring(cid, 4, length(cid))"))
         .otherwise(F.col("cid")))
    .withColumn("bdate",
        F.when(F.col("bdate") > F.current_date(), None).otherwise(F.col("bdate")))
    .withColumn("gen",
        F.when(F.upper(F.trim("gen")).isin("F", "FEMALE"), "Female")
         .when(F.upper(F.trim("gen")).isin("M", "MALE"), "Male")
         .otherwise("n/a"))
)

silver_erp_cust.write.format("delta").mode("overwrite").saveAsTable("silver.erp_cust_az12")
silver_erp_cust.show(5)

StatementMeta(, c427f369-47bc-4533-aa4d-0d042d3457bf, 9, Finished, Available, Finished, False)

+----------+----------+------+
|       cid|     bdate|   gen|
+----------+----------+------+
|AW00011000|1971-10-06|  Male|
|AW00011001|1976-05-10|  Male|
|AW00011002|1971-02-09|  Male|
|AW00011003|1973-08-14|Female|
|AW00011004|1979-08-05|Female|
+----------+----------+------+
only showing top 5 rows



In [8]:
df = spark.read.table("bronze.erp_loc_a101")

silver_erp_loc = (
    df.withColumn("cid", F.regexp_replace("cid", "-", ""))
    .withColumn("cntry",
        F.when(F.trim("cntry") == "DE", "Germany")
         .when(F.trim("cntry").isin("US", "USA"), "United States")
         .when(F.trim("cntry").isNull() | (F.trim("cntry") == ""), "n/a")
         .otherwise(F.trim("cntry")))
)

silver_erp_loc.write.format("delta").mode("overwrite").saveAsTable("silver.erp_loc_a101")
silver_erp_loc.show(5)

StatementMeta(, c427f369-47bc-4533-aa4d-0d042d3457bf, 10, Finished, Available, Finished, False)

+----------+---------+
|       cid|    cntry|
+----------+---------+
|AW00011000|Australia|
|AW00011001|Australia|
|AW00011002|Australia|
|AW00011003|Australia|
|AW00011004|Australia|
+----------+---------+
only showing top 5 rows



In [9]:
silver_erp_cat = spark.read.table("bronze.erp_px_cat_g1v2")
silver_erp_cat.write.format("delta").mode("overwrite").saveAsTable("silver.erp_px_cat_g1v2")
silver_erp_cat.show(5)

StatementMeta(, c427f369-47bc-4533-aa4d-0d042d3457bf, 11, Finished, Available, Finished, False)

+-----+-----------+-----------------+-----------+
|   id|        cat|           subcat|maintenance|
+-----+-----------+-----------------+-----------+
|AC_BR|Accessories|       Bike Racks|        Yes|
|AC_BS|Accessories|      Bike Stands|         No|
|AC_BC|Accessories|Bottles and Cages|         No|
|AC_CL|Accessories|         Cleaners|        Yes|
|AC_FE|Accessories|          Fenders|         No|
+-----+-----------+-----------------+-----------+
only showing top 5 rows



** Gold layer **

In [10]:
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

StatementMeta(, c427f369-47bc-4533-aa4d-0d042d3457bf, 12, Finished, Available, Finished, False)

DataFrame[]

In [11]:
# --- dim_customers ---
cust = spark.read.table("silver.crm_cust_info")
erp_cust = spark.read.table("silver.erp_cust_az12")
erp_loc = spark.read.table("silver.erp_loc_a101")

dim_customers = (
    cust.alias("ci")
    .join(erp_cust.alias("ca"), F.col("ci.cst_key") == F.col("ca.cid"), "left")
    .join(erp_loc.alias("la"), F.col("ci.cst_key") == F.col("la.cid"), "left")
    .select(
        F.col("ci.cst_id").alias("customer_id"),
        F.col("ci.cst_key").alias("customer_number"),
        F.col("ci.cst_firstname").alias("first_name"),
        F.col("ci.cst_lastname").alias("last_name"),
        F.col("la.cntry").alias("country"),
        F.col("ci.cst_marital_status").alias("marital_status"),
        F.coalesce(F.col("ci.cst_gndr"), F.col("ca.gen"), F.lit("n/a")).alias("gender"),
        F.col("ca.bdate").alias("birthdate"),
        F.col("ci.cst_create_date").alias("create_date")
    )
    .withColumn("customer_key", F.monotonically_increasing_id())
)

dim_customers.write.format("delta").mode("overwrite").saveAsTable("gold.dim_customers")
dim_customers.show(5)

StatementMeta(, c427f369-47bc-4533-aa4d-0d042d3457bf, 13, Finished, Available, Finished, False)

+-----------+---------------+----------+---------+---------+--------------+------+----------+-----------+------------+
|customer_id|customer_number|first_name|last_name|  country|marital_status|gender| birthdate|create_date|customer_key|
+-----------+---------------+----------+---------+---------+--------------+------+----------+-----------+------------+
|      11000|     AW00011000|       Jon|     Yang|Australia|       Married|  Male|1971-10-06| 2025-10-06|           0|
|      11003|     AW00011003|   Christy|      Zhu|Australia|        Single|Female|1973-08-14| 2025-10-06|           1|
|      11005|     AW00011005|     Julio|     Ruiz|Australia|        Single|  Male|1976-08-01| 2025-10-06|           2|
|      11006|     AW00011006|     Janet|  Alvarez|Australia|        Single|Female|1976-12-02| 2025-10-06|           3|
|      11007|     AW00011007|     Marco|    Mehta|Australia|       Married|  Male|1969-11-06| 2025-10-06|           4|
+-----------+---------------+----------+--------

In [12]:
prd = spark.read.table("silver.crm_prd_info")
cat = spark.read.table("silver.erp_px_cat_g1v2")

dim_products = (
    prd.alias("p")
    .join(cat.alias("c"), F.col("p.cat_id") == F.col("c.id"), "left")
    .select(
        F.col("p.prd_id").alias("product_id"),
        F.col("p.prd_key").alias("product_number"),
        F.col("p.prd_nm").alias("product_name"),
        F.col("p.cat_id").alias("category_id"),
        F.col("c.cat").alias("category"),
        F.col("c.subcat").alias("subcategory"),
        F.col("c.maintenance"),
        F.col("p.prd_cost").alias("cost"),
        F.col("p.prd_line").alias("product_line"),
        F.col("p.prd_start_dt").alias("start_date")
    )
    .withColumn("product_key", F.monotonically_increasing_id())
)

dim_products.write.format("delta").mode("overwrite").saveAsTable("gold.dim_products")
dim_products.show(5)

StatementMeta(, c427f369-47bc-4533-aa4d-0d042d3457bf, 14, Finished, Available, Finished, False)

+----------+--------------+--------------------+-----------+-----------+-----------+-----------+----+------------+----------+-----------+
|product_id|product_number|        product_name|category_id|   category|subcategory|maintenance|cost|product_line|start_date|product_key|
+----------+--------------+--------------------+-----------+-----------+-----------+-----------+----+------------+----------+-----------+
|       210|    FR-R92B-58|HL Road Frame - B...|      CO_RF| Components|Road Frames|        Yes|   0|        Road|2003-07-01|          0|
|       211|    FR-R92R-58|HL Road Frame - R...|      CO_RF| Components|Road Frames|        Yes|   0|        Road|2003-07-01|          1|
|       212|     HL-U509-R|Sport-100 Helmet-...|      AC_HE|Accessories|    Helmets|        Yes|  12| Other Sales|2011-07-01|          2|
|       213|     HL-U509-R|Sport-100 Helmet-...|      AC_HE|Accessories|    Helmets|        Yes|  14| Other Sales|2012-07-01|          3|
|       214|     HL-U509-R|Sport-1

In [13]:
sales = spark.read.table("silver.crm_sales_details")
dim_cust = spark.read.table("gold.dim_customers")
dim_prod = spark.read.table("gold.dim_products")

fact_sales = (
    sales.alias("s")
    .join(dim_cust.alias("c"), F.col("s.sls_cust_id") == F.col("c.customer_id"), "left")
    .join(dim_prod.alias("p"), F.col("s.sls_prd_key") == F.col("p.product_number"), "left")
    .select(
        F.col("s.sls_ord_num").alias("order_number"),
        F.col("c.customer_key"),
        F.col("p.product_key"),
        F.col("s.sls_order_dt").alias("order_date"),
        F.col("s.sls_ship_dt").alias("ship_date"),
        F.col("s.sls_due_dt").alias("due_date"),
        F.col("s.sls_sales").alias("sales_amount"),
        F.col("s.sls_quantity").alias("quantity"),
        F.col("s.sls_price").alias("price")
    )
)

fact_sales.write.format("delta").mode("overwrite").saveAsTable("gold.fact_sales")

# Check for remaining NULLs and row count
print("Total rows:", fact_sales.count())
print("NULL product_key:", fact_sales.filter(F.col("product_key").isNull()).count())
print("NULL customer_key:", fact_sales.filter(F.col("customer_key").isNull()).count())

StatementMeta(, c427f369-47bc-4533-aa4d-0d042d3457bf, 15, Finished, Available, Finished, False)

Total rows: 89852
NULL product_key: 0
NULL customer_key: 0


In [14]:
fact_sales.filter(F.col("customer_key").isNull() | F.col("product_key").isNull()).count()

StatementMeta(, c427f369-47bc-4533-aa4d-0d042d3457bf, 16, Finished, Available, Finished, False)

0

In [15]:
fact_sales.filter(F.col("product_key").isNull()).show()

StatementMeta(, c427f369-47bc-4533-aa4d-0d042d3457bf, 17, Finished, Available, Finished, False)

+------------+------------+-----------+----------+---------+--------+------------+--------+-----+
|order_number|customer_key|product_key|order_date|ship_date|due_date|sales_amount|quantity|price|
+------------+------------+-----------+----------+---------+--------+------------+--------+-----+
+------------+------------+-----------+----------+---------+--------+------------+--------+-----+

